# manual-chain-forward-and-back — worked example 2: Manual Forward and Backward Through sigmoid → square-error

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `manual-chain-forward-and-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The sigmoid function σ(x) = 1/(1+e^−x) has a convenient backward form: its derivative is σ(x)·(1−σ(x)), which can be expressed entirely in terms of its output `out`. Chaining it with a squaring operation illustrates how cached outputs flow backward: square_back uses its input, and sigmoid_back uses its output.

## Worked solution

**Step 1 — forward pass: a → b → c.**
We compute `b = sigmoid(a)` then `c = b²`. We cache both `b` and `c`.

**Step 2 — square_back: compute dL/db.**
The derivative of `b²` with respect to `b` is `2b`. So `dL/db = dL/dc * 2*b`. We use the input `b` here.

**Step 3 — sigmoid_back: compute dL/da.**
The derivative of sigmoid is `sigmoid(a) * (1 - sigmoid(a)) = b * (1 - b)`. So `dL/da = dL/db * b * (1 - b)`. We use the cached output `b` (= sigmoid(a)) from the forward pass — this is the key efficiency of the sigmoid backward: no re-evaluation of the exponential is needed.

**Step 4 — verify against autograd.**
We rebuild the chain with `requires_grad=True` and confirm that PyTorch's `.backward()` gives the same gradient as our manual computation.

In [ ]:
import torch as t

def sigmoid_back(grad_out, out, x):
    """d/dx sigmoid(x) = sigmoid(x)*(1 - sigmoid(x)) = out*(1 - out)"""
    return grad_out * out * (1.0 - out)

def square_back(grad_out, out, x):
    """d/dx x^2 = 2x; uses input x"""
    return grad_out * 2.0 * x

def manual_sigmoid_square_chain(a, dL_dc):
    # Forward
    b = t.sigmoid(a)    # b = sigmoid(a)
    c = b ** 2          # c = b^2
    # Backward (reverse order)
    dL_db = square_back(dL_dc, c, b)      # uses input b
    dL_da = sigmoid_back(dL_db, b, a)     # uses output b
    return b, c, dL_db, dL_da

# Run it
t.manual_seed(17)
a_val = t.tensor([-1.0, 0.0, 2.0, -3.0])
dL_dc_val = t.ones(4)

b, c, dL_db, dL_da = manual_sigmoid_square_chain(a_val, dL_dc_val)
print(f"a       = {a_val}")
print(f"b=σ(a)  = {b.round(decimals=4)}")
print(f"c=b²    = {c.round(decimals=4)}")
print(f"dL/db   = {dL_db.round(decimals=4)}")
print(f"dL/da (manual)  = {dL_da.round(decimals=4)}")

# Verify with autograd
a_ag = a_val.clone().requires_grad_(True)
c_ag = t.sigmoid(a_ag) ** 2
loss = (c_ag * dL_dc_val).sum()
loss.backward()
print(f"dL/da (autograd) = {a_ag.grad.round(decimals=4)}")
print(f"Match: {t.allclose(dL_da, a_ag.grad, atol=1e-5)}")